In [1]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TerraIncognito-sudo/311_Agent_MMAI.git"
REPO_DIR = Path("/kaggle/working/311_Agent_MMAI")

if REPO_DIR.exists():
    print("Repo already exists — pulling latest main...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Git HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

# Install dependencies WITHOUT -U to avoid Kaggle pyarrow/matplotlib conflicts
req_file = REPO_DIR / "requirements.txt"
if req_file.exists():
    print("Installing requirements.txt...")
    subprocess.run(["pip", "install", "-q", "-r", str(req_file)], check=False)

# Ensure transformers / accelerate / pyyaml are available for local encoder models
subprocess.run(["pip", "install", "-q", "transformers>=4.48.0", "accelerate", "pyyaml"], check=False)

import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


Cloning repository...


Cloning into '/kaggle/working/311_Agent_MMAI'...


Working directory: /kaggle/working/311_Agent_MMAI
Git HEAD: 72db45a
Installing requirements.txt...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.


PyTorch: 2.11.0+cu130 | CUDA available: True


In [2]:
import glob
import os
import shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/311_Agent_MMAI")
os.chdir(REPO_DIR)

# Use in-memory fixture config instead of Supabase
os.environ["USE_MEMORY_STORE"] = "1"

# 1. Inspect .env.example so we know which API key(s) the repo expects
env_example = REPO_DIR / ".env.example"
if env_example.exists():
    print("--- .env.example ---")
    print(env_example.read_text(encoding="utf-8").strip())
    print("--------------------\n")

# 2. Load API keys from Kaggle Secrets (if available)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key_name in ["OPENROUTER_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
        try:
            val = secrets.get_secret(key_name)
            if val:
                os.environ[key_name] = val
                print(f"Loaded Kaggle Secret: {key_name} (len={len(val)})")
        except Exception:
            pass
except ImportError:
    print("Not running in Kaggle Secrets environment; using existing os.environ keys.")

# 3. Auto-locate locked test_heldout.jsonl in /kaggle/input
TEST_DEST = REPO_DIR / "data/test/test_heldout.jsonl"
TEST_DEST.parent.mkdir(parents=True, exist_ok=True)

heldout_matches = (
    glob.glob("/kaggle/input/**/test_heldout.jsonl", recursive=True)
    or glob.glob("/kaggle/input/**/*heldout*.jsonl", recursive=True)
)
if heldout_matches:
    src_test = Path(heldout_matches[0])
    shutil.copy2(src_test, TEST_DEST)
    n_lines = sum(1 for line in TEST_DEST.open(encoding="utf-8") if line.strip())
    print(f"Copied locked test set: {src_test} -> {TEST_DEST} ({n_lines} cases)")
elif TEST_DEST.exists():
    n_lines = sum(1 for line in TEST_DEST.open(encoding="utf-8") if line.strip())
    print(f"Using existing test set at {TEST_DEST} ({n_lines} cases)")
else:
    print("WARNING: test_heldout.jsonl not found in /kaggle/input/!")
    print("Attached input files (first 20):", glob.glob("/kaggle/input/**/*", recursive=True)[:20])

# 4. Auto-locate fine-tuned model folders and symlink into src/models/fine_tuning/
FT_DIR = REPO_DIR / "src/models/fine_tuning"
FT_DIR.mkdir(parents=True, exist_ok=True)

model_configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
linked_models = []
for cfg in model_configs:
    model_dir = Path(cfg).parent
    lower_path = str(model_dir).lower()
    if "modernbert" in lower_path:
        target = FT_DIR / "ModernBERT-base"
    elif "distilroberta" in lower_path:
        target = FT_DIR / "distilroberta"
    else:
        continue

    if target.is_symlink() or target.exists():
        if target.is_symlink():
            target.unlink()
        else:
            shutil.rmtree(target)
    os.symlink(model_dir, target)
    linked_models.append((target.name, str(model_dir)))

print("\nLinked fine-tuned models in src/models/fine_tuning/:")
for name, src in linked_models:
    print(f"  - {name} -> {src}")
if not linked_models:
    print("  (None auto-linked. Check /kaggle/input/ or ensure weights are unzipped.)")


--- .env.example ---
# Copy to .env and fill in. .env is git-ignored: keys live ONLY there.

# --- Supabase (tickets, handoffs, interaction_events, knowledge base) ---
# Project Settings -> API in the Supabase dashboard.
SUPABASE_URL=https://your-project-ref.supabase.co
# Server-side secret key (service_role / sb_secret_...). Never expose to the browser.
SUPABASE_SERVICE_KEY=
# Set to 1 to run without Supabase using an in-memory store (dev/tests only;
# nothing persists across restarts).
USE_MEMORY_STORE=0

# --- Ollama (local LLM server) ---
OLLAMA_BASE_URL=http://localhost:11434

# --- OpenRouter ---
# No key here on purpose: each user enters their own key in the browser.
# It is held in page memory only and sent with each chat request.
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
# Shown to OpenRouter as the calling app (optional)
OPENROUTER_APP_URL=
OPENROUTER_APP_NAME=Gilead 311 Agent

# --- Web server (python run.py serve) ---
HOST=127.0.0.1
PORT=8311

# --- Docker (docker co

In [3]:
import subprocess

print("=== Registered Harness Setups ===")
res = subprocess.run(
    ["python", "-m", "src.eval.harness", "setups"],
    capture_output=True,
    text=True,
)
print(res.stdout)
if res.stderr:
    print("STDERR:\n", res.stderr)


=== Registered Harness Setups ===
classifier: keyword_stub, distilroberta_lora, modernbert_lora
agent: ollama_qwen3_4b, haiku, sonnet
judge: sonnet_judge



### Run the Eval Harness (`classify`)
Review the setup names printed above and update `SETUPS_TO_RUN` below if any name differs from what is registered in `src/eval/harness.py`.

Each run writes:
- `results/<run_id>.csv` *(safe to commit to GitHub)*
- `results/manifests/<run_id>.json` *(safe to commit to GitHub)*
- `results/cache/<run_id>/` *(private — share with Shawn/Abi for `make reproduce`)*
- `results/events/<run_id>_events.jsonl` *(private — share with Shawn/Abi)*


In [4]:
import subprocess
import sys
import time
from pathlib import Path

TEST_PATH = "data/test/test_heldout.jsonl"
TEST_NAME = "heldout"

# UPDATE THESE NAMES if `python -m src.eval.harness setups` printed different setup IDs
SETUPS_TO_RUN = [
    "keyword_stub",
    "distilroberta",
    "ModernBERT-base",
    # Uncomment / adjust to match the exact prompted baseline names from Cell 4:
    # "haiku_zero_shot",
    # "haiku_few_shot",
    # "sonnet_zero_shot",
    # "sonnet_few_shot",
]

assert Path(TEST_PATH).exists(), f"Missing {TEST_PATH}. Attach the locked test set in Cell 3."

run_status = {}
for setup in SETUPS_TO_RUN:
    print("=" * 72)
    print(f"Running harness classify | setup={setup} | test={TEST_NAME}")
    print("=" * 72)
    t0 = time.time()
    cmd = [
        sys.executable, "-m", "src.eval.harness",
        "classify",
        "--setup", setup,
        "--test", TEST_PATH,
        "--name", TEST_NAME,
    ]
    proc = subprocess.run(cmd, text=True)
    elapsed = time.time() - t0
    run_status[setup] = {"returncode": proc.returncode, "elapsed_s": round(elapsed, 1)}
    print(f"--> Finished {setup} (rc={proc.returncode}) in {elapsed:.1f}s\n")

print("=== Run Summary ===")
for setup, info in run_status.items():
    status = "OK" if info["returncode"] == 0 else f"FAILED (rc={info['returncode']})"
    print(f"  {setup:<22} : {status} ({info['elapsed_s']}s)")


AssertionError: Missing data/test/test_heldout.jsonl. Attach the locked test set in Cell 3.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

RESULTS_DIR = Path("results")
ROUTE_CODES = [
    "SW-BIN", "SW-MISS", "AN-DEAD", "AN-INJ", "RD-POT",
    "TR-TREE", "PS-PROP", "IL-DUMP", "EMERG", "OOS",
]

csv_files = sorted(RESULTS_DIR.glob("*_heldout_*.csv"))
print(f"Found {len(csv_files)} heldout result CSV(s):")
for p in csv_files:
    print(" ", p.name)

def parse_latency_ms(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    s = str(val).strip()
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list) and parsed:
            return float(np.sum(parsed))
        return float(parsed)
    except Exception:
        try:
            return float(s)
        except Exception:
            return np.nan

summary_rows = []
loaded_dfs = {}

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    if df.empty or "expected_code" not in df.columns or "predicted_code" not in df.columns:
        continue

    setup_name = str(df["setup"].iloc[0]) if "setup" in df.columns else csv_path.stem
    run_id = str(df["run_id"].iloc[0]) if "run_id" in df.columns else csv_path.stem
    loaded_dfs[run_id] = df

    y_true = df["expected_code"].fillna("MISSING").astype(str)
    y_pred = df["predicted_code"].fillna("MISSING").astype(str)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=ROUTE_CODES, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=ROUTE_CODES, average="weighted", zero_division=0)

    emerg_mask = y_true == "EMERG"
    emerg_recall = (y_pred[emerg_mask] == "EMERG").mean() if emerg_mask.any() else np.nan
    oos_f1 = f1_score(y_true == "OOS", y_pred == "OOS", zero_division=0)

    latencies = df["turn_latencies_ms"].apply(parse_latency_ms) if "turn_latencies_ms" in df.columns else pd.Series(dtype=float)
    mean_lat = latencies.mean()
    p95_lat = latencies.quantile(0.95) if latencies.notna().any() else np.nan

    total_cost = pd.to_numeric(df.get("cost_usd", 0), errors="coerce").fillna(0).sum()
    n_errors = df["error"].fillna("").astype(str).str.strip().ne("").sum() if "error" in df.columns else 0

    summary_rows.append({
        "run_id": run_id,
        "setup": setup_name,
        "n_cases": len(df),
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "emerg_recall": round(emerg_recall, 4) if pd.notna(emerg_recall) else np.nan,
        "oos_f1": round(oos_f1, 4),
        "mean_latency_ms": round(mean_lat, 1) if pd.notna(mean_lat) else np.nan,
        "p95_latency_ms": round(p95_lat, 1) if pd.notna(p95_lat) else np.nan,
        "total_cost_usd": round(total_cost, 4),
        "errors": int(n_errors),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
summary_df.to_csv("/kaggle/working/nb06_classifier_summary.csv", index=False)
display(summary_df)


In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

if not loaded_dfs:
    print("No result CSVs loaded to plot.")
else:
    # 1. Per-class F1 breakdown table
    per_class = {}
    for run_id, df in loaded_dfs.items():
        setup_name = str(df["setup"].iloc[0])
        f1s = f1_score(
            df["expected_code"].astype(str),
            df["predicted_code"].fillna("MISSING").astype(str),
            labels=ROUTE_CODES,
            average=None,
            zero_division=0,
        )
        per_class[setup_name] = np.round(f1s, 4)

    per_class_df = pd.DataFrame(per_class, index=ROUTE_CODES)
    per_class_df.to_csv("/kaggle/working/nb06_per_class_f1.csv")
    print("=== Per-Class F1 by Setup ===")
    display(per_class_df)

    # 2. Normalized confusion matrices grid
    n_models = len(loaded_dfs)
    ncols = min(2, n_models)
    nrows = math.ceil(n_models / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (run_id, df) in zip(axes, loaded_dfs.items()):
        setup_name = str(df["setup"].iloc[0])
        cm = confusion_matrix(
            df["expected_code"].astype(str),
            df["predicted_code"].fillna("MISSING").astype(str),
            labels=ROUTE_CODES,
            normalize="true",
        )
        sns.heatmap(
            cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=ROUTE_CODES, yticklabels=ROUTE_CODES,
            vmin=0, vmax=1, cbar=False, ax=ax,
        )
        ax.set_title(f"{setup_name} (n={len(df)})")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Expected")

    for ax in axes[n_models:]:
        ax.axis("off")

    plt.tight_layout()
    out_png = "/kaggle/working/nb06_confusion_matrices.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved confusion matrices to {out_png}")


In [ ]:
import zipfile
from pathlib import Path

RESULTS_DIR = Path("results")
GITHUB_ZIP = Path("/kaggle/working/nb06_github_commit.zip")
PRIVATE_ZIP = Path("/kaggle/working/nb06_private_reproduce_bundle.zip")

run_ids = sorted(p.stem for p in RESULTS_DIR.glob("*_heldout_*.csv"))
print(f"Packaging {len(run_ids)} heldout run(s): {run_ids}\n")

# 1. GitHub-safe bundle (ONLY results/<run_id>.csv and results/manifests/<run_id>.json)
with zipfile.ZipFile(GITHUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for rid in run_ids:
        csv_p = RESULTS_DIR / f"{rid}.csv"
        man_p = RESULTS_DIR / "manifests" / f"{rid}.json"
        if csv_p.exists():
            zf.write(csv_p, arcname=str(csv_p))
        if man_p.exists():
            zf.write(man_p, arcname=str(man_p))

print(f"Created GITHUB-SAFE zip: {GITHUB_ZIP} ({GITHUB_ZIP.stat().st_size / 1024:.1f} KB)")
with zipfile.ZipFile(GITHUB_ZIP) as zf:
    for name in zf.namelist():
        print("  [GitHub OK] ", name)

# 2. Private reproduce bundle (cache/<run_id>/ and events/<run_id>_events.jsonl — DO NOT COMMIT)
with zipfile.ZipFile(PRIVATE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for rid in run_ids:
        cache_dir = RESULTS_DIR / "cache" / rid
        if cache_dir.exists():
            for f in cache_dir.rglob("*"):
                if f.is_file():
                    zf.write(f, arcname=str(f))
        ev_p = RESULTS_DIR / "events" / f"{rid}_events.jsonl"
        if ev_p.exists():
            zf.write(ev_p, arcname=str(ev_p))

print(f"\nCreated PRIVATE reproduce zip: {PRIVATE_ZIP} ({PRIVATE_ZIP.stat().st_size / 1024:.1f} KB)")
print("Share this zip privately with Shawn & Abi (do NOT push to GitHub).")


In [ ]:
import subprocess

print("=== Testing `python -m src.eval.harness reproduce` ===")
res = subprocess.run(
    ["python", "-m", "src.eval.harness", "reproduce"],
    capture_output=True,
    text=True,
)
print(res.stdout)
if res.stderr:
    print("STDERR:\n", res.stderr)
print("Exit code:", res.returncode)
